### prepair modules and bases settings

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets, linear_model
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix,  classification_report, log_loss
from sklearn.preprocessing import StandardScaler
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.svm import SVC
# from sklearn.ensemble import RandomForestClassifier
from scipy.stats import norm
import scipy.io
import re
import itertools

import os
from os.path import join
import contextlib
from copy import deepcopy
import imp 
import time 
import sys

import pickle
from pdb import set_trace

from IPython.display import clear_output, display

In [2]:
# Add the directory containing your modules to the Python path
sys.path.append(os.path.abspath(os.path.join('..', 'ses2_modelstims')))

# load local functions
import stim_io
import stim_io_plotting
import vtc
import bvbabel

/home/jorvhar/miniconda3/envs/predlis/lib/python3.8/site-packages/nilearn/glm/__init__.py:55: FutureWarning: The nilearn.glm module is experimental. It may change in any future release of Nilearn.
  warn('The nilearn.glm module is experimental. '


In [4]:
## LOADING GRID

# Load from MAT file
variables = scipy.io.loadmat('/media/jorvhar/Data8T/MRIData/timing data/grid_parameters_python.mat')

# Extract individual variables
tunsteps = variables['tunsteps']
freqstep = variables['freqstep']
subsample = variables['subsample']
mustep = variables['mustep']
muarray_bins = variables['muarray_bins']
muarray = variables['muarray']
fwhm = variables['fwhm']
octgrid = variables['octgrid']
sigmagrid = variables['sigmagrid']
pref_range = variables['pref_range']
sharp_range_fwhm = variables['sharp_range_fwhm']
sharp_range = variables['sharp_range']

### Load Regression Parameters for easier parsing

In [5]:
### --- LOCATION OPTIONS ---

# file location
mridat_dir = '/media/jorvhar/Data8T/MRIData/PreProc'

# tonotopy and mask filenames
tonotopy_vmp = 'prf_permutations_for_s2.vmp'
mask_fn = 'gm-subcortical.msk'
pp_dir = lambda pp, ses : f'S{pp:02d}_SES{ses}'

### --- PARTICIPANT OPTIONS ---

# session of interest
ses = 2

# variable that may be different per participant
ppz = [1,2,3,4,5,6,7,8,9,10]
n_splitsz = [6,6,5,5,5,5,5,5,5,5]               # 12 runs > 10:2cross, 6 fold, splits used per pp (for variable length option)
n_runz = [12,12,10,10,10,10,10,10,10,10]        # number of runs
startpp = 1

### --- SET THEORIE REGRESSION MODELS ---

# for full 3 sets we need 7 sets
models = ['base',
          'adaptation',
          'prediction',
          'base_U_adaptation',
          'base_U_prediction',
          'adaptation_U_prediction',
          'base_U_adaptation_U_prediction']
# models = ['prediction',
#           'base_U_adaptation',
#           'base_U_adaptation_U_prediction']
# if we want to use sets we only need 3 models 
##models=['base_U_adaptation', 'prediction', 'base_U_adaptation_U_prediction']

## REGRESSORS IN MODELS ##
model_regressors = {'base':       ['raw_acti', 'onoff'], 
                    'adaptation': ['raw_adapt' ],      # adaptation
                    'prediction': ['pred_prob',   # voxelwise prior liklihood
                                   'surprisal',
                                   'precision']   # global prior surprise
                   } 
# if we want to add adapted activation
# model_regressors['adaptation'] += ['adapt_activ']

# set combination of regressors
model_regressors.update({'base_U_adaptation':             model_regressors['base']+
                                                          model_regressors['adaptation'],
                        'base_U_prediction':              model_regressors['base']+
                                                          model_regressors['prediction'], 
                        'adaptation_U_prediction':        model_regressors['adaptation']+
                                                          model_regressors['prediction'], 
                        'base_U_adaptation_U_prediction': model_regressors['base']+
                                                          model_regressors['adaptation']+
                                                          model_regressors['prediction']})

## ART-ANOVA Prep
We save our regression results (ROI based averages), to set in long format and later load in R - where we do the `Aligned Rank Transformed ANOVA` within `ARTAnova.ipynb`

In [15]:
scores_fn = 'ANTS_scores_pw_uw_lwo_tempsmooth' # 'scores_prec', 'IdealObserver_scores_pres_tempsmooth', 'IdealObserver_scores_pres'
# scores_fn = 'ANTS_IdealObserver_scores_pw_uw_tempsmooth'
corrected_fn = 'uncorrected' # 'uncorrected'
cv_fn = 'cv(median)' #'cv(median)' #'cv(median)' # 'noncv'
handle_negatives = 'aubuc_remove'
threshold_p = 0.05

average_df = pd.read_pickle(join(mridat_dir, f'R2S_{scores_fn}_{handle_negatives}_{corrected_fn}_{cv_fn}(p{threshold_p:.0e})-binned.pickle'))
average_df_temp = average_df.loc[average_df['roi'].str.contains('combined')]

layers = ['Layer 1', 'Layer 2', 'Layer 3']
layernames = ['Deep', 'Middle', 'Superficial']
    
# df for roi
roi_df = average_df_temp.loc[(average_df_temp['layer'].isin(layers))].copy()

# Reshape from wide to long format using all non-ID columns
long_df = pd.melt(
    roi_df,
    id_vars=['pp', 'roi', 'layer'], # Columns to keep as identifiers
    var_name='Model',               # All other columns ('base', 'base_u_adaptation', etc.) go here
    value_name='Response'           # The actual data values
)

# save data 
long_df.to_csv(join(mridat_dir, 'fMRI_art_data.csv'), index=False)

## Draining vein analsyis and plotting
The following code parses 3-way shared variances and overall performance of combined model - as sanity check for draining veins

In [ ]:
def get_star_line(stars, position, sy1='∗', sy2='⋅'):
    star_counts = [s.count(sy1) + s.count(sy2) for s in stars]
    line_parts = []
    for i, count in enumerate(star_counts):
        if i == position:
            original_stars = ''.join(c for c in stars[i] if c in f'{sy1}{sy2}')
            line_parts.append(original_stars)
        else:
            line_parts.append(' ' * count)
    # Now join without adding any extra spaces
    return ' '.join(line_parts)

def get_symbol_line(stars, symbol='-', sy1='∗', sy2='⋅'):
    star_counts = [s.count(sy1) + s.count(sy2) for s in stars]
    if len(stars) <= 1:
        return ''
    return f'{symbol}'.join(' ' * i for i in star_counts)

# sig values # †
# * : p<0.05
# ** : p<0.01
# *** : p<0.001
# † : p_uncorrect<0.05
sigs = {}
sigs['HG_combined'] = {'sig_12': [], 'sig_23': [], 'sig_13': []}
sigs['PP_combined'] = {'sig_12': [], 'sig_23': [], 'sig_13': []}
sigs['PT_combined'] = {'sig_12': [], 'sig_23': [], 'sig_13': []}
sigs['aSTG_combined'] = {'sig_12': [], 'sig_23': [], 'sig_13': []}
sigs['pSTG_combined'] = {'sig_12': [], 'sig_23': [], 'sig_13': []}
sigs_colors = {'prediction_prior∗': '#72aea0',
               'prediction_error∗': '#8ca0ca',
               'prediction_prior_n_prediction_error∗': '#7FA7B5',
               'prediction_prior∗∗': '#72aea0',
               'prediction_error∗∗': '#8ca0ca',
               'prediction_prior_n_prediction_error∗∗': '#7FA7B5',
               'prediction_prior∗∗∗': '#72aea0',
               'prediction_error∗∗∗': '#8ca0ca',
               'prediction_prior_n_prediction_error∗∗∗': '#7FA7B5',               
               'prediction_prior†': '#72aea0',
               'prediction_error†': '#8ca0ca',
               'prediction_prior_n_prediction_error†': '#7FA7B5',               
               'prediction_prior⋅': '#72aea0',
               'prediction_error⋅': '#8ca0ca',
               'prediction_prior_n_prediction_error⋅': '#7FA7B5'}
sigs_posx = {'sig_12': 0.32,
               'sig_23': 0.68,
               'sig_13': 0.5}
sigs_posy = {'sig_12': -0.2675,
               'sig_23': -0.2675,
               'sig_13': -0.3375}
sigs_posy = {'sig_12': -0.2475,
               'sig_23': -0.2475,
               'sig_13': -0.3175}

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from os.path import join
from matplotlib.transforms import blended_transform_factory
import matplotlib.patheffects as path_effects

def smart_jitter(values, center=0, x_range=0.15, min_dist=1.5):
    """Jitter points horizontally around a center while avoiding overlap in both axes."""
    positions = []
    y_spacing = np.std(values) / 100  # small fudge factor
    for y in values:
        tries = 0
        while True:
            x = np.random.uniform(center - x_range, center + x_range)
            too_close = False
            for (x0, y0) in positions:
                if np.sqrt((x - x0)**2 + ((y - y0)/min_dist)**2) < 0.03:
                    too_close = True
                    break
            if not too_close or tries > 100:
                positions.append((x, y))
                break
            tries += 1
    x_jittered, y_jittered = zip(*positions)
    return np.array(x_jittered)

scores_fn = 'ANTS_scores_pw_uw_er_lwo_tempsmooth' 
corrected_fn = 'uncorrected'
cv_fn = 'cv(median)' 
threshold_p = 0.05

# outline
outline = True
show_ylabel = False

# set capwith
errwidth = 0.1  # Adjust this value to change cap width

# set rois
rois = ['HG_combined', 'PP_combined', 'PT_combined', 'aSTG_combined', 'pSTG_combined']
for area in rois:

    average_df_temp = pd.read_pickle(join(mridat_dir, f'R2S_{scores_fn}_{handle_negatives}_{corrected_fn}_{cv_fn}(p{threshold_p:.0e})-binned.pickle'))

    layers = ['Layer 1', 'Layer 2', 'Layer 3']
    filtered_df = average_df_temp.loc[(average_df_temp['roi'] == area) & (average_df_temp['layer'].isin(layers))]

    plt.figure(figsize=(2, 2), dpi=300)

    # old including baseline
#     colors = ['#b3b4b5', '#72aea0', '#8ca0ca']
#     colors_light = ['#D9DADA', '#B9D7D0', '#C6D0E5']
#     labels = ['Baseline Model', 'Priors', 'Errors']
#     columns = ['base*', 'prediction_prior*', 'prediction_error*']
#     x_offsets = [-0.2, 0.0, 0.2]
    
    colors = ['#B56979', '#906F9A']
    colors_light = ['#D09FAA', '#AC93B4']
    labels = ['Full model', 'Fully shared variance']
    columns = ['base_u_adaptation_u_prediction', 'base_n_adaptation_n_prediction']
#     columns = ['base_n_prediction_prior*', 'base_n_prediction_error*']
    x_offsets = [-0.15, 0.15]
    
    
    
 
    # get peak vals
    peaks = [filtered_df.groupby('layer')['base_u_adaptation_u_prediction'].mean().max(),
             filtered_df.groupby('layer')['base_n_adaptation_n_prediction'].mean().max()]
    
#     colors = ['#72aea0', '#8ca0ca']
#     colors_light = ['#B9D7D0', '#C6D0E5']
#     labels = ['Priors', 'Errors']
#     columns = ['prediction_prior*', 'prediction_error*']
#     x_offsets = [-0.1, 0.1]

    for idx, (col, label, color, colorligh, offset) in enumerate(zip(columns, labels, colors, colors_light, x_offsets)):
        for i, layer in enumerate(layers):
            layer_data = filtered_df.loc[filtered_df['layer'] == layer, col] / peaks[idx]
            x_pos = i + offset

            # scatter individual datapoints
            x_jitter = smart_jitter(layer_data.to_numpy(), center=x_pos, x_range=0.07, min_dist=2.0)
            plt.scatter(x_jitter, layer_data, s=30,  color=colorligh, edgecolors='none')

            # CI line using user-defined bootstrap function
            ci_low, ci_high = stats.bootstrap_ci(layer_data.to_numpy())
            median_val = np.mean(layer_data)

            if outline:

                # display middle
                plt.plot(x_pos, median_val, 'o', color=color, markersize=7.5, 
                         markeredgewidth=1.0, markeredgecolor='grey', zorder=3)

                # Vertical CI bar with matching capstyle
                plt.plot([x_pos, x_pos], [ci_low, ci_high], color='grey', lw=5.5, zorder=2, solid_capstyle='butt')  # outline
                plt.plot([x_pos, x_pos], [ci_low, ci_high], color=color, lw=3.5, zorder=3, solid_capstyle='butt')   # main line

                # Bottom cap
                plt.plot([x_pos - errwidth/2, x_pos + errwidth/2], [ci_low, ci_low], color='grey', lw=3, zorder=2)
                plt.plot([x_pos - errwidth/2, x_pos + errwidth/2], [ci_low, ci_low], color=color, lw=1.5, zorder=3)

                # Top cap
                plt.plot([x_pos - errwidth/2, x_pos + errwidth/2], [ci_high, ci_high], color='grey', lw=3, zorder=2)
                plt.plot([x_pos - errwidth/2, x_pos + errwidth/2], [ci_high, ci_high], color=color, lw=1.5, zorder=3)
            
            else:
                
                # display middle
                plt.plot(x_pos, median_val, 'o', color=color, markersize=7.5)
            
                # draw CI bar and median dot
                plt.plot([x_pos, x_pos], [ci_low, ci_high], color=color, lw=4.5, solid_capstyle='butt')
                plt.plot(x_pos, median_val, 'o', color=color, markersize=7.5, mew=0, markeredgecolor=color)

                # draw horizontal caps aligned exactly at x_pos
                plt.plot([x_pos - errwidth/2, x_pos + errwidth/2], [ci_low, ci_low], color=color, lw=1.5)
                plt.plot([x_pos - errwidth/2, x_pos + errwidth/2], [ci_high, ci_high], color=color, lw=1.5)
                


    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(3)
    ax.spines['bottom'].set_linewidth(3)

    ax.tick_params(axis='x', labelsize=9)
    ax.tick_params(axis='y', labelsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2f}'))
    ax.set_xticks(range(len(layers)))
    ax.set_xticklabels(['Deep', 'Middle', 'Superficial'], fontsize=9)

#     ylimmax = ci_low * 1.10
#     ylimmin = ci_high * 1.10
#     ax.set_ylim(ylimmin, ylimmax)
    
    ylimmax = (np.max(filtered_df[columns]/peaks) + 0.15)
    ylimmin = (np.min(filtered_df[columns]/peaks) - 0.25)
    ax.set_ylim(ylimmin, ylimmax)

    # horizontal line at 0
    plt.axhline(0, linestyle='--', color='#919191', alpha=1, linewidth=1.5, zorder=0)

    for idx, label in enumerate(labels):
        plt.plot([], [], color=colors[idx], lw=4, label=label)
    

 # === Draw layer comparison lines below plot, outside data space ===

    # lookup sig values
    sig_lookup = sigs.get(area, {})

    # Get axis and figure transform to position below axis area
    trans = blended_transform_factory(ax.transData, ax.transAxes)

    y_pos_main = -0.24  # controls vertical position under the x-axis (play with this value)
    spacing = 0.07     # vertical spacing for non-overlapping long line

    # Comparison: Deep–Middle (0–1)
    if len(sig_lookup['sig_12']) > 0: # check if any sig
        ax.plot([0.05, 0.95], [y_pos_main, y_pos_main], transform=trans, color='grey', lw=1.5, clip_on=False)

    # Comparison: Middle–Superficial (1–2)
    if len(sig_lookup['sig_23']) > 0: # check if any sig
        ax.plot([1.05, 1.95], [y_pos_main, y_pos_main], transform=trans, color='grey', lw=1.5, clip_on=False)

    # Comparison: Deep–Superficial (0–2), offset slightly lower
    if len(sig_lookup['sig_13']) > 0: # check if any sig
        ax.plot([0.05, 1.95], [y_pos_main - spacing, y_pos_main - spacing], transform=trans, color='grey', lw=1.5, clip_on=False)

    # Optional: extend figure bottom margin to make space
    plt.subplots_adjust(bottom=0.25)

    
    ## set significance marking
    for comp in ['sig_12', 'sig_23', 'sig_13']:
        stars = sig_lookup.get(comp, [])
        for st_idx, star in enumerate(stars):
            constructtxt = get_star_line(stars, st_idx)
            
            txt = ax.text(sigs_posx[comp], sigs_posy[comp], constructtxt, horizontalalignment='center', verticalalignment='center', 
                          transform=ax.transAxes, fontsize=12, fontweight='bold', color=sigs_colors[star], fontname='monospace')
            txt.set_path_effects([
                            path_effects.Stroke(linewidth=1.5, foreground='black'),
                            path_effects.Normal()
                            ])
        # add commas
        commas = get_symbol_line(stars, symbol='-')
        txt = ax.text(sigs_posx[comp], sigs_posy[comp]+0.01, commas, horizontalalignment='center', verticalalignment='center', 
                          transform=ax.transAxes, fontsize=12, color='black', fontweight='bold', fontname='monospace')

    # adjust labeling
    if show_ylabel:
        plt.ylabel('Relative Signal\nProfile (a.u.)', fontsize=10)
    else:
        ax.set_ylabel(None)
        ax.yaxis.set_label_coords(-0.1, 0.5)  # pushes invisible label space out, optional
        area = f'splitpred_{area}_noYlabel'
    
    
    # save
    outname_base = join('visualisation', f'layerresults_dv_{area}' if outline else f'layerresults_{area}')
#     plt.savefig(f'{outname_base}.png', bbox_inches='tight', dpi=300)
    plt.savefig(f'{outname_base}.pdf', bbox_inches='tight')
#     plt.savefig(f'{outname_base}.jpg', bbox_inches='tight', dpi=300)
        
    plt.show()


as well as the legends for these plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

colors = ['#B56979', '#906F9A']
labels = ['Stimulus Preference ∪ Repetition Suppression ∪ Expectation', 'Stimulus Preference ∩ Repetition Suppression ∩ Expectation']

# Create dummy lines with grey outlines
legend_handles = []
for color, label in zip(colors, labels):
    line, = plt.plot([], [], color=color, lw=4, label=label,
                     path_effects=[path_effects.Stroke(linewidth=6, foreground='grey', capstyle='round'),
                                   path_effects.Normal()])
    legend_handles.append(line)

# Create a new figure just for the legend
fig_legend = plt.figure(figsize=(4.5, 1.2))  # wider figure for horizontal
ax = fig_legend.add_subplot(111)

# Horizontal legend: ncol=number of items
legend = ax.legend(handles=legend_handles, loc='center', frameon=False, ncol=len(labels),
                   handlelength=2.5, handletextpad=0.8, columnspacing=1.5)

# Hide axes
ax.axis('off')

# # Save legend as separate figure
fig_legend.savefig(join('visualisation','legend_only_draining_veins.pdf'), bbox_inches='tight')

plt.show()

## Load blocked regressor effect
Next we load in the control regression that - instead of zscoring bold time courses per run - add blocked regressors to explain this portion of variance. For this you first run the code within `regression)blocked_effects` for displaying the results with the code below.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# set ROIS
rois = ['HG', 'PP', 'PT', 'aSTG', 'pSTG']
hemispheres = ['LH', 'RH', 'combined']
layers = ['Layer 1', 'Layer 2', 'Layer 3']

# create lamdas for dict
roisLambda = lambda hs: f'{hs}_auditory.voi'
boundaryLambda = lambda hs: f'{hs}_boundary.voi'
depthLambda = lambda hs: f'winning_WMGM_{hs}_D1-3.voi'

# selmodel: 'base_U_adaptation_U_prediction'  
#           'base_U_prediction_prior_U_prediction_error'
# filename: 'ANTS_scores_pw_uw_lwo_tempsmooth'  
#           'ANTS_scores_pw_uw_lwo'
#           'ANTS_IdealObserver_scores_pw_uw_lwo_tempsmooth'
#           'ANTS_predictors_split_prec_pw_lwo_tempsmooth'
#           'ANTS_IdealObserver_predictors_split_prec_lwo_tempsmooth'

selmodel = 'base_U_adaptation_U_prediction' #'base_U_adaptation_U_prediction' # base_U_prediction_prior_U_prediction_error
threshold_p = 0.05
scores_fn = 'ANTS_scores_pw_uw_er_lwo_tempsmooth' # or 'scores_tempsmooth' or 'IdealObserver_scores_tempsmooth' 
# scores_fn = 'ANTS_IdealObserver_scores_pw_uw_lwo_tempsmooth' # or 'scores_tempsmooth' or 'IdealObserver_scores_tempsmooth' 
ppz = [1,2,3,4,5,6,7,8,9,10]

# load current scores for blocked effect
scores_fn_blocked = 'ANTS_scores_blocked_effects_tempsmooth'

# other parameters
correct_r2s = False
cv = True
cvmethod = 'median'
handle_negatives = 'aubuc_remove' #'zero', 'zero_partials', 'remove', 'aubuc_remove'

# get full grida
grid = list(itertools.product(rois, hemispheres))

# initialize an empty DataFrame for storing means
average_df = pd.DataFrame()

# loop over participants
for pp_idx, pp in enumerate(ppz):
    
    print(f'Participant: {pp}')
    
    # open pickle of vois
    with open(join(mridat_dir, pp_dir(pp, 2), 'allVOIs_inset_winning.pickle'), 'rb') as handle:
        voi_dict = pickle.load(handle)
    # and of results
    with open(join(mridat_dir, pp_dir(pp, 2), f'Betas/{scores_fn}.pickle'), 'rb') as handle:
        scores = pickle.load(handle)
    # and of results blocked
    with open(join(mridat_dir, pp_dir(pp, 2), f'Betas/{scores_fn_blocked}.pickle'), 'rb') as handle:
        scores_blocked = pickle.load(handle)
        
    # sellect models
    models = list(scores.keys())[:-1]
        
    # do another sweep without layers
    # get indexes
    indx_dict_nl = stim_io.generate_voxel_indices(hemispheres, rois, voi_dict, scores['indexes'], depthLambda=depthLambda)
    
    # Loop over ROIs
    for i, gp in enumerate(grid):
        
        # get combined index
        roi = f'{gp[0]}_{gp[1]}'
        
        # Get subgroup of good performing voxels
        subidx = np.where(scores[selmodel]['non-cv']['p_value'][indx_dict_nl[roi]] < threshold_p)
        newidx = indx_dict_nl[roi][subidx]

        ## translate coords
        # create coordinate tuples for blocked data
        coords_blocked = list(zip(scores_blocked['indexes'][0],
                                  scores_blocked['indexes'][1],
                                  scores_blocked['indexes'][2]))
        # dictionary: coordinate -> index
        blocked_lookup = {coord: i for i, coord in enumerate(coords_blocked)}
        coords_selected = list(zip(scores['indexes'][0][newidx],
                                   scores['indexes'][1][newidx],
                                   scores['indexes'][2][newidx]))
        newidx_blocked = np.array([
            blocked_lookup[c] for c in coords_selected if c in blocked_lookup
        ])
        
        # get varpart labels
        models = ['base_U_adaptation_U_prediction']
        
        # sellect blocked using same voxels as before
        score_sel = {model: scores_blocked[model]['non-cv']['raw_scores'][newidx_blocked] for model in models}   #noncv

        # append to average dataframe
        mean_series = pd.DataFrame()
        mean_series['score'] = [score_sel['base_U_adaptation_U_prediction'].mean()]
        mean_series['pp'] = pp
        mean_series['roi'] = roi
        mean_series['layer'] = 'combined'
        
        print(score_sel['base_U_adaptation_U_prediction'].mean())
        
        average_df = average_df._append(mean_series, ignore_index=True)

Lastly a very quick (demo) presentation of the mean r2 explained per roi, considering signficant voxels

In [ ]:
average_df

rois = ['HG_combined', 'PP_combined', 'PT_combined', 'aSTG_combined', 'pSTG_combined']

for roi in rois:
    mn = average_df.loc[average_df['roi'] == roi, 'score'].mean()
    std = average_df.loc[average_df['roi'] == roi, 'score'].std()
    
    print(f'roi: {roi}, has mean: {mn:.3f}, and std: {std:.3f}')